#### 문제 01

서로 다른 네 가지 예측 작업에서 모델의 마지막 출력값을 어떤 형태로 변환해야 하는지 결정하기.

**상황**
1. 설비의 고장 여부처럼 두 가지 중 하나를 예측.
2. 사진 속 동물이 고양이, 강아지, 토끼 중 하나인지 예측.
3. 하나의 사진에 '야간', '실내', '사람 있음' 태그가 동시에 붙을 수 있는지 예측.

**입력 데이터**
```python
binary_logits = torch.tensor([-1.2, 0.7])
animal_logits = torch.tensor([1.2, 2.0, 0.4])
tag_logits = torch.tensor([0.8, -0.5, 1.4])
```

**요구사항**
1. 각 입력 텐서를 생성.
2. 이진 분류에는 Sigmoid 적용.
3. 서로 배타적인 세 클래스 분류에는 Softmax 적용.
4. 여러 태그가 동시에 참일 수 있는 경우에는 각 출력에 Sigmoid 적용.
5. 작업별 최종 출력값을 구분하여 출력.

In [1]:
import torch
from torch import nn, optim

In [9]:
binary_logits = torch.tensor([-1.2, 0.7])
animal_logits = torch.tensor([1.2, 2.0, 0.4])
tag_logits = torch.tensor([0.8, -0.5, 1.4])

binary_result = torch.sigmoid(binary_logits)
print('binary :',binary_result)

animal_result = torch.softmax(animal_logits, dim=0)
print('animal :',animal_result, 'animal index :',animal_result.argmax())

tag_result = torch.sigmoid(tag_logits)

print('tag :',tag_result)


binary : tensor([0.2315, 0.6682])
animal : tensor([0.2721, 0.6056, 0.1223]) animal index : tensor(1)
tag : tensor([0.6900, 0.3775, 0.8022])


#### 문제 02

**입력 데이터**
```python
values = [-3.0, -1.0, 0.0, 1.0, 3.0]
```

**요구사항**
1. 동일한 값으로 두 개의 입력 텐서를 생성하고 각각 `requires_grad=True` 설정.
2. 첫 번째 텐서에는 ReLU를 적용하고, 두 번째 텐서에는 Tanh를 적용.
3. 각 출력 텐서의 모든 원소를 더해 스칼라 결과 생성.
4. 두 결과에 각각 `backward()` 실행.
5. ReLU와 Tanh의 출력값 및 입력 텐서의 Gradient를 비교하여 출력.

In [10]:
values = [-3.,-1,0,1,3]

a = torch.tensor(values, requires_grad=True)
b = torch.tensor(values, requires_grad=True)

relu = torch.relu(a)
tanh = torch.tanh(b)

relu_sum = relu.sum()
tanh_sum = tanh.sum()

relu_sum.backward()
tanh_sum.backward()

print(relu)
print(tanh)
print(a.grad)
print(b.grad)


tensor([0., 0., 0., 1., 3.], grad_fn=<ReluBackward0>)
tensor([-0.9951, -0.7616,  0.0000,  0.7616,  0.9951], grad_fn=<TanhBackward0>)
tensor([0., 0., 0., 1., 1.])
tensor([0.0099, 0.4200, 1.0000, 0.4200, 0.0099])


#### 문제 03

하나의 상품에 여러 가지 특징 태그가 동시에 붙을 수 있는 다중 레이블(Multi-label) 분류 결과 만들기.

**상황**
상품 3개에 대해 다음 4가지 태그를 예측함.
- 0번: 할인 상품
- 1번: 신상품
- 2번: 재고 부족
- 3번: 무료 배송

**입력 데이터**
```python
logits = torch.tensor([
    [1.2, -0.8, 0.4, 2.0],
    [-0.5, 1.7, 1.1, -1.0],
    [0.9, 0.2, -1.2, 0.6]
])
```

태그별 분류 기준은 `[0.6, 0.5, 0.7, 0.4]`임.

**요구사항**
1. Logits 텐서 생성.
2. 각 태그에 독립적으로 Sigmoid 적용.
3. 태그별 기준값 텐서를 생성.
4. 예측 확률과 기준값을 비교하여 Boolean 예측 결과 생성.
5. 상품별 태그 예측 결과와 각 상품의 예측 태그 개수 출력.

In [14]:
logits = torch.tensor([[1.2,-0.8,0.4,2],[-0.5,1.7,1.1,-1.0],[0.9,0.2,-1.2,0.6]])
tag = torch.tensor([0.6,0.5,0.7,0.4])

probablilty = torch.sigmoid(logits)

result = probablilty >= tag
print(result)
print(result.sum(dim=1))

tensor([[ True, False, False,  True],
        [False,  True,  True, False],
        [ True,  True, False,  True]])
tensor([2, 2, 3])


#### 문제 04

서로 다른 예측 작업에 적합한 손실함수를 선택하고, 각 작업의 손실을 한 번에 계산하기.

**상황 및 데이터**
1. 배송 소요 시간 예측: 회귀 문제
2. 불량 여부 예측: Sigmoid를 거친 확률이 이미 주어진 이진 분류 문제
3. 결제 이상 여부 예측: 모델의 원래 Logits가 주어진 이진 분류 문제
4. 문의 유형 예측: 4개 유형 중 하나를 고르는 다중 분류 문제

```python
time_prediction = torch.tensor([2.5, 4.0, 5.5])
time_target = torch.tensor([3.0, 3.5, 6.0])

defect_probability = torch.tensor([0.85, 0.30, 0.65])
defect_target = torch.tensor([1.0, 0.0, 0.0])

payment_logits = torch.tensor([-1.0, 1.5, 0.4])
payment_target = torch.tensor([0.0, 1.0, 0.0])

inquiry_logits = torch.tensor([
    [0.2, 1.5, 0.7, -0.3],
    [1.8, 0.4, 0.1, 0.5]
])
inquiry_target = torch.tensor([1, 0])
```

**요구사항**
1. 각 데이터 텐서를 생성.
2. 배송 소요 시간에는 `nn.L1Loss()` 적용.
3. 불량 여부 확률에는 `nn.BCELoss()` 적용.
4. 결제 이상 Logits에는 `nn.BCEWithLogitsLoss()` 적용.
5. 문의 유형 Logits에는 `nn.CrossEntropyLoss()` 적용.
6. 네 가지 손실값을 각각 출력.

In [16]:
time_prediction = torch.tensor([2.5, 4.0, 5.5])
time_target = torch.tensor([3.0, 3.5, 6.0])

defect_probability = torch.tensor([0.85, 0.30, 0.65])
defect_target = torch.tensor([1.0, 0.0, 0.0])

payment_logits = torch.tensor([-1.0, 1.5, 0.4])
payment_target = torch.tensor([0.0, 1.0, 0.0])

inquiry_logits = torch.tensor([
    [0.2, 1.5, 0.7, -0.3],
    [1.8, 0.4, 0.1, 0.5]
])
inquiry_target = torch.tensor([1, 0])

loss_time = nn.L1Loss()(time_prediction, time_target)
loss_defect = nn.BCELoss()(defect_probability, defect_target)
loss_payment = nn.BCEWithLogitsLoss()(payment_logits, payment_target)
loss_inquiry = nn.CrossEntropyLoss()(inquiry_logits, inquiry_target)

print(loss_time)
print(loss_defect)
print(loss_payment)
print(loss_inquiry)

tensor(0.5000)
tensor(0.5230)
tensor(0.4759)
tensor(0.5834)


#### 문제 05

여러 출력 항목 중 중요도가 서로 다른 회귀 모델에서 항목별 중요도를 반영한 손실값 계산하기.

**상황**
한 모델이 상품의 가격과 배송 시간을 동시에 예측함. 가격 오차에 배송 시간 오차보다 3배 높은 중요도를 부여하려고 함.

**입력 데이터**
```python
prediction = torch.tensor([
    [105.0, 2.5],
    [190.0, 4.5],
    [150.0, 3.0]
])

target = torch.tensor([
    [100.0, 3.0],
    [200.0, 4.0],
    [140.0, 3.5]
])
```

**요구사항**
1. `nn.MSELoss(reduction='none')`으로 원소별 제곱 오차 계산.
2. `mean(dim=0)`을 이용하여 가격과 배송 시간의 평균 오차를 각각 계산.
3. 항목별 가중치 `[3.0, 1.0]` 생성.
4. 항목별 평균 오차에 가중치를 곱한 뒤 합산.
5. 가중치를 반영하기 전과 후의 손실값 출력.

In [23]:
prediction = torch.tensor([
    [105.0, 2.5],
    [190.0, 4.5],
    [150.0, 3.0]
])

target = torch.tensor([
    [100.0, 3.0],
    [200.0, 4.0],
    [140.0, 3.5]
])

loss = nn.MSELoss(reduction='none')(prediction, target)
loss_mean = loss.mean(dim=0)
weight = torch.tensor([3.0,1.0])
result = (loss_mean*weight).sum()
print(loss_mean.sum())
print(result)

tensor(75.2500)
tensor(225.2500)


#### 문제 06

SGD Optimizer를 사용하여 하나의 파라미터를 두 번 업데이트하기.

**초기 조건**
- 파라미터 w: 1.5
- 목표값 target: 6.0
- 예측식: `prediction = 2 * w`
- 손실함수: `(prediction - target) ** 2`
- Learning Rate: 0.05

**요구사항**
1. w를 `nn.Parameter()`로 생성.
2. `torch.optim.SGD()`로 Optimizer 생성.
3. 총 2회 반복 학습.
4. 매 반복마다 `optimizer.zero_grad()` 실행.
5. 예측값과 손실 계산.
6. `loss.backward()`로 Gradient 계산.
7. `optimizer.step()`으로 파라미터 업데이트.
8. 각 반복의 손실값, Gradient, 업데이트 후 w 출력.

In [25]:
w = nn.Parameter(torch.tensor(1.5))
target = 6.0

optimizer = optim.SGD([w], lr=0.05)

for epoch in range(2):
    optimizer.zero_grad()
    prediction = 2*w
    loss = (prediction-target)**2
    loss.backward()
    optimizer.step()

    print(f'Epoch : {epoch+1}')
    print('손실값 :', loss.item())
    print('Gradient :', w.grad.item())
    print('w :', w.item())


Epoch : 1
손실값 : 9.0
Gradient : -12.0
w : 2.0999999046325684
Epoch : 2
손실값 : 3.2400007247924805
Gradient : -7.200000762939453
w : 2.4600000381469727


#### 문제 07

CrossEntropyLoss가 계산하는 값을 Softmax와 로그를 이용하여 직접 재현하고, PyTorch 결과와 비교하기.

**입력 데이터**
```python
logits = torch.tensor([
    [1.4, 0.2, 2.1],
    [0.1, 1.7, 0.5],
    [2.0, 0.8, 0.3]
])

target = torch.tensor([2, 1, 0])
```

**요구사항**
1. Logits와 정답 텐서 생성.
2. `torch.softmax(logits, dim=1)`로 클래스별 확률 계산.
3. 각 행에서 실제 정답 클래스에 해당하는 확률만 선택.
4. 선택한 확률에 `torch.log()`를 적용하고 음수 부호를 붙인 뒤 평균 계산.
5. `nn.CrossEntropyLoss()`의 결과와 직접 계산한 결과 비교.
6. `torch.allclose()`로 두 결과가 같은지 확인.

In [34]:
logits = torch.tensor([[1.4,0.2,2.1],[0.1,1.7,0.5],[2.0,0.8,0.3]])
target = torch.tensor([2,1,0])

a = torch.softmax(logits, dim=1)
b = -torch.log(a[torch.arange(3),target]).mean()
c = nn.CrossEntropyLoss()(logits, target)
print(b)
print(c)

print(torch.allclose(b,c))

tensor(0.4335)
tensor(0.4335)
True


#### 문제 08

두 개의 데이터 묶음에서 계산한 Gradient가 누적되는 현상을 확인하고, `optimizer.zero_grad()`의 역할 검증.

**초기 조건**
- 파라미터 w: 1.0
- 첫 번째 손실식: `(2*w - 5) ** 2`
- 두 번째 손실식: `(w - 4) ** 2`

**요구사항**
1. w를 `nn.Parameter()`로 생성하고 SGD Optimizer 생성.
2. `optimizer.zero_grad()` 실행 후 첫 번째 손실을 계산하고 `backward()` 실행.
3. 첫 번째 손실에서 계산된 Gradient 출력.
4. Gradient를 초기화하지 않고 두 번째 손실에 `backward()` 실행.
5. 누적된 Gradient 출력.
6. `optimizer.zero_grad()`로 Gradient 초기화.
7. 두 번째 손실을 다시 계산하고 `backward()` 실행.
8. 초기화 후 Gradient와 누적 Gradient의 차이 출력.

In [36]:
w = nn.Parameter(torch.tensor(1.0))
optimizer = optim.SGD([w], lr=0.01)
optimizer.zero_grad()
loss1 = (2*w-5)**2
loss1.backward()

print(w.grad)

loss2 = (w-4)**2
loss2.backward()
print(w.grad)

optimizer.zero_grad()
loss2 = (w-4)**2
loss2.backward()
print(w.grad)

tensor(-12.)
tensor(-18.)
tensor(-6.)


#### 문제 09

Optimizer의 실행 순서를 잘못 작성했을 때 파라미터가 업데이트되지 않는 이유 확인하기.

**초기 조건**
- 파라미터 w: 2.0
- 목표값 target: 7.0
- 손실식: `(3*w - target) ** 2`
- Optimizer: SGD
- Learning Rate: 0.1

**요구사항**
1. 동일한 초기값을 가진 파라미터 두 개 생성.
2. 각각 SGD Optimizer 생성.
3. 첫 번째 파라미터는 `zero_grad()` → 손실 계산 → `backward()` → `step()` 순서로 실행.
4. 두 번째 파라미터는 `zero_grad()` → 손실 계산 → `step()` → `backward()` 순서로 실행.
5. 두 파라미터의 업데이트 전후 값과 Gradient 출력.
6. 두 번째 파라미터가 첫 번째와 다르게 동작한 이유를 코드 주석으로 작성.

In [ ]:
w1 = nn.Parameter(torch.tensor(2.0))
w2 = nn.Parameter(torch.tensor(2.0))
target = torch.tensor(7.0)

optimizer1 = optim.SGD([w1], lr=0.1)
optimizer2 = optim.SGD([w2], lr=0.1)

print('w1 업뎃 전 :', w1.item())
optimizer1.zero_grad()
loss1 = (3*w1 - target)**2
loss1.backward()
optimizer1.step()

print('w2 업뎃 전 :', w2.item())
optimizer2.zero_grad()
loss2 = (3*w2 - target)**2
optimizer2.step()
loss2.backward()

print('w1 업뎃 후 :', w1.item())
print('w2 업뎃 후 :', w2.item())
print('w1 Gradient :',w1.grad.item())
print('w2 Gradient :',w2.grad.item())

#backward를 늦게 해서 step()에 반영되지 않음(Gradient에는 반영됨)

w1 업뎃 전 : 2.0
w2 업뎃 전 : 2.0
w1 업뎃 후 : 2.5999999046325684
w2 업뎃 후 : 2.0
w1 Gradient : -6.0
w2 Gradient : -6.0
